In [1]:
from pathlib import Path
import sys

if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: c:\code\testing\LLMs_Robustness_Under_Distractions


In [2]:
import json
import pandas as pd
from IPython.display import display

from src.generation import (
    load_candidates_from_jsonl,
    select_final_base_examples,
    select_base_examples_exact,
    count_by_task_and_status,
)

from src.base_dataset import (
    build_base_dataset,
    build_dataset_summary,
    save_jsonl,
    save_json,
)
from src.validation import validate_dataset

In [3]:
DATA_DIR = PROJECT_ROOT / "data"
BASE_DIR = DATA_DIR / "base"

BASE_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)

BASE_DIR: c:\code\testing\LLMs_Robustness_Under_Distractions\data\base


In [4]:
# Load the manually reviewed candidate pool
REVIEWED_DIR = DATA_DIR / "reviewed"
reviewed_candidates_path = REVIEWED_DIR / "reviewed_candidates.jsonl"

all_candidates = load_candidates_from_jsonl(str(reviewed_candidates_path))

print("Loaded reviewed candidates:", len(all_candidates))
print(json.dumps(count_by_task_and_status(all_candidates), indent=2, ensure_ascii=False))

Loaded reviewed candidates: 250
{
  "single_label_classification": {
    "approved": 2,
    "rejected": 1,
    "pending": 47
  },
  "multi_label_classification": {
    "pending": 50
  },
  "information_extraction": {
    "pending": 50
  },
  "rule_based_transformation": {
    "pending": 50
  },
  "extractive_qa": {
    "pending": 50
  }
}


In [5]:
# Build exactly 250 base examples:
# approved examples are used first, then pending examples fill the remaining slots.
selected_candidates = select_base_examples_exact(all_candidates, n_per_task=50)

selected_counts = {}
selected_status_counts = {"approved": 0, "pending": 0, "rejected": 0}

for example in selected_candidates:
    selected_counts[example.task_name] = selected_counts.get(example.task_name, 0) + 1
    selected_status_counts[example.review_status] = (
        selected_status_counts.get(example.review_status, 0) + 1
    )

print("Selected counts by task:", selected_counts)
print("Selected counts by review status:", selected_status_counts)

base_records = build_base_dataset(selected_candidates)
print("Total selected candidates:", len(selected_candidates))
print("Total base records:", len(base_records))

if selected_status_counts["rejected"] > 0:
    print("=" * 100)
    print("WARNING: this 250-example dataset includes rejected examples as fallback.")
    print("It is a temporary draft dataset and should not be treated as the final reviewed base set.")
    print("=" * 100)
elif selected_status_counts["pending"] > 0:
    print("=" * 100)
    print("WARNING: this 250-example dataset is a draft build.")
    print("It contains pending examples because the approved-only reviewed set is not complete yet.")
    print("=" * 100)
else:
    print("Final approved-only 250-example dataset assembled successfully.")

Selected counts by task: {'extractive_qa': 50, 'information_extraction': 50, 'multi_label_classification': 50, 'rule_based_transformation': 50, 'single_label_classification': 50}
Selected counts by review status: {'approved': 2, 'pending': 247, 'rejected': 1}
Total selected candidates: 250
Total base records: 250
It is a temporary draft dataset and should not be treated as the final reviewed base set.


In [6]:
base_records = build_base_dataset(selected_candidates)
print("Total approved selected candidates:", len(selected_candidates))
print("Total base records:", len(base_records))

Total approved selected candidates: 250
Total base records: 250


In [7]:
# Inspect a few sample records
for record in base_records[:5]:
    print("=" * 100)
    print(json.dumps(record, indent=2, ensure_ascii=False))

{
  "example_id": "qa_200",
  "task_name": "extractive_qa",
  "template_name": "schedule_passage",
  "instruction": "Answer the question using an exact span from the passage.",
  "input_data": {
    "passage": "Alice Smith arrived in Rome on 2024-01-15 for a training session.",
    "question": "Where did Alice Smith arrive?"
  },
  "gold_output": {
    "answer": "Rome"
  },
  "metadata": {
    "answer_field": "location"
  }
}
{
  "example_id": "qa_201",
  "task_name": "extractive_qa",
  "template_name": "schedule_passage",
  "instruction": "Answer the question using an exact span from the passage.",
  "input_data": {
    "passage": "Alice Smith arrived in Milan on 2024-03-21 for a training session.",
    "question": "Where did Alice Smith arrive?"
  },
  "gold_output": {
    "answer": "Milan"
  },
  "metadata": {
    "answer_field": "location"
  }
}
{
  "example_id": "qa_202",
  "task_name": "extractive_qa",
  "template_name": "schedule_passage",
  "instruction": "Answer the question u

In [8]:
# Build a dataset summary
dataset_summary = build_dataset_summary(base_records)
print(json.dumps(dataset_summary, indent=2, ensure_ascii=False))

{
  "total_records": 250,
  "counts_by_task": {
    "extractive_qa": 50,
    "information_extraction": 50,
    "multi_label_classification": 50,
    "rule_based_transformation": 50,
    "single_label_classification": 50
  },
  "instructions_by_task": {
    "extractive_qa": [
      "Answer the question using an exact span from the passage."
    ],
    "information_extraction": [
      "Extract the fields person, date, and location from the text."
    ],
    "multi_label_classification": [
      "Assign all applicable topic labels from {finance, health, politics, sports, tech}. Return them alphabetically."
    ],
    "rule_based_transformation": [
      "Convert the text to lowercase.",
      "Remove all punctuation from the text.",
      "Remove all words longer than 6 characters from the text.",
      "Replace every number in the text with <NUM>."
    ],
    "single_label_classification": [
      "Classify the sentiment of the text using exactly one label from {positive, negative, neut

In [9]:
# Convert the summary to a dataframe view
dataset_summary = build_dataset_summary(base_records)
print(json.dumps(dataset_summary, indent=2, ensure_ascii=False))

{
  "total_records": 250,
  "counts_by_task": {
    "extractive_qa": 50,
    "information_extraction": 50,
    "multi_label_classification": 50,
    "rule_based_transformation": 50,
    "single_label_classification": 50
  },
  "instructions_by_task": {
    "extractive_qa": [
      "Answer the question using an exact span from the passage."
    ],
    "information_extraction": [
      "Extract the fields person, date, and location from the text."
    ],
    "multi_label_classification": [
      "Assign all applicable topic labels from {finance, health, politics, sports, tech}. Return them alphabetically."
    ],
    "rule_based_transformation": [
      "Convert the text to lowercase.",
      "Remove all punctuation from the text.",
      "Remove all words longer than 6 characters from the text.",
      "Replace every number in the text with <NUM>."
    ],
    "single_label_classification": [
      "Classify the sentiment of the text using exactly one label from {positive, negative, neut

In [10]:
# Show the instructions used for each task
for task_name, instructions in dataset_summary["instructions_by_task"].items():
    print("=" * 100)
    print("TASK:", task_name)
for instruction in instructions:
    print("-", instruction)
    print("NUM_UNIQUE_INSTRUCTIONS:", len(instructions))
    print()

TASK: extractive_qa
TASK: information_extraction
TASK: multi_label_classification
TASK: rule_based_transformation
TASK: single_label_classification
- Classify the sentiment of the text using exactly one label from {positive, negative, neutral}.
NUM_UNIQUE_INSTRUCTIONS: 1



In [11]:
# Run full dataset validation
validation_report = validate_dataset(base_records)
print(json.dumps(validation_report, indent=2, ensure_ascii=False))

{
  "is_valid": true,
  "total_records": 250,
  "num_records_with_issues": 0,
  "record_issues": {},
  "dataset_level_issues": [
    "instruction_inconsistency:rule_based_transformation:num_unique_instructions=4"
  ],
  "fatal_dataset_level_issues": [],
  "issue_counts": {
    "instruction_inconsistency:rule_based_transformation:num_unique_instructions=4": 1
  }
}


In [12]:
# Stop here if validation failed
if not validation_report["is_valid"]:
    raise ValueError("Validation failed. Inspect validation_report before proceeding.")

print("Validation passed. The clean base dataset is stable.")

Validation passed. The clean base dataset is stable.


In [13]:
# Inspect any record-level issues if needed
if validation_report["record_issues"]:
    for example_id, issues in list(validation_report["record_issues"].items())[:20]:
        print("=" * 100)
        print("EXAMPLE ID:", example_id)
        print("ISSUES:", issues)
else:
    print("No record-level issues found.")

No record-level issues found.


In [14]:
# Save the clean base dataset
base_jsonl_path = BASE_DIR / "base_examples.jsonl"
base_json_path = BASE_DIR / "base_examples.json"
validation_json_path = BASE_DIR / "validation_report.json"
summary_json_path = BASE_DIR / "dataset_summary.json"

save_jsonl(base_records, str(base_jsonl_path))
save_json(base_records, str(base_json_path))
save_json(validation_report, str(validation_json_path))
save_json(dataset_summary, str(summary_json_path))

print("Saved:")
print("-", base_jsonl_path)
print("-", base_json_path)
print("-", validation_json_path)
print("-", summary_json_path)


Saved:
- c:\code\testing\LLMs_Robustness_Under_Distractions\data\base\base_examples.jsonl
- c:\code\testing\LLMs_Robustness_Under_Distractions\data\base\base_examples.json
- c:\code\testing\LLMs_Robustness_Under_Distractions\data\base\validation_report.json
- c:\code\testing\LLMs_Robustness_Under_Distractions\data\base\dataset_summary.json


In [15]:
# Reload the JSONL file as a final sanity check
reloaded_records = []
with open(base_jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        reloaded_records.append(json.loads(line))

print("Reloaded records:", len(reloaded_records))
print("First reloaded record:")
print(json.dumps(reloaded_records[0], indent=2, ensure_ascii=False))

Reloaded records: 250
First reloaded record:
{
  "example_id": "qa_200",
  "gold_output": {
    "answer": "Rome"
  },
  "input_data": {
    "passage": "Alice Smith arrived in Rome on 2024-01-15 for a training session.",
    "question": "Where did Alice Smith arrive?"
  },
  "instruction": "Answer the question using an exact span from the passage.",
  "metadata": {
    "answer_field": "location"
  },
  "task_name": "extractive_qa",
  "template_name": "schedule_passage"
}


We built the clean base dataset of 250 examples from the manually reviewed candidate pool, selecting only approved examples and keeping 50 examples for each of the five task families. The examples were stored in a structured format containing the task type, standardized instruction, input data, and gold output. Before moving on to distraction generation, the clean base set was validated for label correctness, schema consistency, deterministic transformations, exact extractive answer spans, unique IDs, and correct task counts. Only after review-based selection and successful validation was the clean base set considered stable.